# Passing-Lane Analysis (ground, lofted, through balls)

Reads the same detection metadata used by the other pipeline notebooks. For a chosen passer
(default: nearest player to the ball), computes:

- **Ground lanes** to every teammate: open/blocked by the **perpendicular distance from each
  defender to the pass line segment** (not distance to the receiver or passer), plus the
  nearest defender's exact shift-needed-to-block if the lane is open.
- **Lofted alternatives**, only for receivers whose ground lane is blocked: a landing-zone
  pressure estimate (opponents near the receiver's position), scaled by pass distance since a
  longer lofted ball gives defenders more time to converge.
- **Through balls**, only if attack direction is configured (see CONFIG) — a run beyond the
  defensive line, with a lane-clear check, a distance-based "race margin" proxy, and an
  explicit offside-risk flag.

## Calibration: real per-image homography (not grass-outline guessing)
Pitch positions come from a **per-image homography fit to detected pitch-line keypoints**
(same method as `team_pipeline_full.ipynb` / `bird_eye_any_angle.ipynb`), not from fitting a
box to the visible grass region. Fitting a homography to the visible grass region assumes the
frame shows the whole pitch corner-to-corner, which is false for zoomed/panned/tilted broadcast
cameras — it silently stretches whatever's visible over the full 120 x 68 m pitch. The
keypoint-based homography is checked for a low reprojection error and rejected if it isn't
trustworthy — printed as a skip reason, not used with degraded accuracy. A grass-based
`"approx"` mode is still available as a no-model fallback for wide shots that show the whole
pitch, but is off by default.

## Hard requirements this pipeline enforces (not soft suggestions)
- Real-meter distances only make sense with a homography that's actually checked for
  correctness. Any image where calibration fails or looks implausible (most people project off
  the pitch, or collapse onto a line) is **excluded from this analysis entirely** — not run
  with degraded accuracy, skipped outright, and logged as such.
- Through-ball analysis needs attack direction, which isn't derivable from a single image.
  Without `CONFIG["attack_direction"]` set for a team, through-ball output for that team's
  passer is simply omitted — never guessed.
- All proxies here (lofted pressure, through-ball race margin) are explicitly geometric/
  distance-based approximations, not physics or player-speed simulations — a single frame has
  no velocity information. Treat as coaching visualization aids, not predictive certainty.

**Setup:** `pip install opencv-python-headless scikit-learn matplotlib pillow numpy`, then for
keypoint calibration, either `pip install inference-sdk` + a free `ROBOFLOW_API_KEY`, or
`pip install ultralytics` + local pitch-keypoint weights (see CONFIG).

In [ ]:
%pip install -q opencv-python-headless --upgrade

In [ ]:
import os, json, hashlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans

try:
    from inference_sdk import InferenceHTTPClient
except ImportError:
    InferenceHTTPClient = None

In [ ]:
CONFIG = {
    "metadata_paths": {
        "train": "./metadata/train_metadata.jsonl",
        "test": "./metadata/test_metadata.jsonl",
    },
    "output_dir": "./passing_lane_stats",

    "min_person_count": 5,
    "min_players_for_team_split": 4,

    # ---- Pitch calibration ----
    # "keypoints" = true per-image homography from pitch-line keypoints (RECOMMENDED; needs a source below)
    # "approx"    = no model: pitch shape guessed from the grass region. Only OK for wide frames showing the
    #               whole pitch. WRONG for zoomed/panned/tilted broadcast frames — use "keypoints" for those.
    "calibration": "keypoints",
    "min_grass_frac": 0.20,
    "keypoint_weights_path": None,      # local YOLOv8-pose weights trained on the 32 pitch keypoints
    "roboflow_api_key": os.environ.get("ROBOFLOW_API_KEY", ""),
    "keypoint_model_id": "football-field-detection-f07vi/14",
    "keypoint_conf": 0.6,
    "min_keypoints": 6,
    "ransac_px": 6.0,
    "min_pitch_hull_m2": 150.0,
    "min_on_pitch_frac": 0.7,
    "min_spread_m": 3.0,
    "max_outside_m": 15.0,
    "cache_dir": "./keypoint_cache",
    "show_reprojection": True,          # verification step: draw projected pitch lines on the real image

    "interception_radius_m": 2.0,       # defender within this distance of a lane blocks it
    "lane_t_range": [0.1, 0.9],         # excludes defenders just marking the passer/receiver

    "contest_radius_base_m": 4.0,       # lofted landing-zone pressure radius at a 20m pass
    "contest_radius_per_10m": 0.6,      # + this many meters of radius per extra 10m of pass distance
                                         # (longer lofted balls hang longer -> more defenders can converge)

    # Attack direction per team, only needed for through-ball analysis. "increasing_x" means
    # this team attacks toward larger pitch_x; "decreasing_x" toward smaller pitch_x.
    # Leave a team as None to skip through-ball analysis for that team's passer (no guessing).
    "attack_direction": {
        "red": None,
        "blue": None,
    },

    "passer_selection": "nearest_to_ball",   # or an explicit (team, index-within-team) tuple
    "num_verification_images": 5,
}

PERSON_CLASSES = {"player", "goalkeeper", "referee"}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

_any_direction_set = any(v is not None for v in CONFIG["attack_direction"].values())
if not _any_direction_set:
    print("NOTE: no attack_direction configured for either team - through-ball analysis will "
          "be omitted for all images. Ground and lofted-pass analysis are unaffected.")

In [ ]:
# ============================================================
# Load metadata (unchanged pattern from earlier notebooks).
# ============================================================
def load_metadata(path):
    records = []
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — skipping.")
        return records
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if "error" in record:
                continue
            records.append(record)
    return records

## Pitch calibration (keypoints -> homography, or approx grass fallback)

In [ ]:
# ============================================================
# Pitch model in METERS. Same layout/ordering as the Roboflow 32-keypoint football-field model
# (roboflow/sports SoccerPitchConfiguration): 120 x 70 m, penalty box 41 x 20.15, goal box 18.32 x 5.5.
# ============================================================
L, W = 120.0, 70.0
PB_W, PB_L = 41.0, 20.15
GB_W, GB_L = 18.32, 5.5
CC_R, PEN_SPOT = 9.15, 11.0

PITCH_VERTICES = np.array([
    (0, 0), (0, (W-PB_W)/2), (0, (W-GB_W)/2), (0, (W+GB_W)/2), (0, (W+PB_W)/2), (0, W),
    (GB_L, (W-GB_W)/2), (GB_L, (W+GB_W)/2), (PEN_SPOT, W/2),
    (PB_L, (W-PB_W)/2), (PB_L, (W-GB_W)/2), (PB_L, (W+GB_W)/2), (PB_L, (W+PB_W)/2),
    (L/2, 0), (L/2, W/2-CC_R), (L/2, W/2+CC_R), (L/2, W),
    (L-PB_L, (W-PB_W)/2), (L-PB_L, (W-GB_W)/2), (L-PB_L, (W+GB_W)/2), (L-PB_L, (W+PB_W)/2),
    (L-PEN_SPOT, W/2),
    (L-GB_L, (W-GB_W)/2), (L-GB_L, (W+GB_W)/2),
    (L, 0), (L, (W-PB_W)/2), (L, (W-GB_W)/2), (L, (W+GB_W)/2), (L, (W+PB_W)/2), (L, W),
    (L/2-CC_R, W/2), (L/2+CC_R, W/2),
], dtype=np.float32)

_client = None
_local_model = None
def _get_client():
    global _client
    if _client is None:
        if InferenceHTTPClient is None:
            raise RuntimeError("pip install inference-sdk  (or use CONFIG['keypoint_weights_path'] instead)")
        key = CONFIG["roboflow_api_key"] or os.environ.get("ROBOFLOW_API_KEY", "")   # read at call time
        if not key:
            raise RuntimeError("No keypoint source: set ROBOFLOW_API_KEY (env var or CONFIG['roboflow_api_key']) "
                               "or CONFIG['keypoint_weights_path'].")
        _client = InferenceHTTPClient(api_url="https://detect.roboflow.com", api_key=key)
    return _client


def get_keypoints(image_path):
    """Returns list of [x, y, conf] in model order (index = keypoint id). Cached on disk."""
    os.makedirs(CONFIG["cache_dir"], exist_ok=True)
    key = hashlib.md5(os.path.abspath(image_path).encode()).hexdigest()
    cpath = os.path.join(CONFIG["cache_dir"], key + ".json")
    if os.path.exists(cpath):
        with open(cpath) as f:
            return json.load(f)

    kps = []
    if CONFIG["keypoint_weights_path"]:
        global _local_model
        if _local_model is None:
            from ultralytics import YOLO
            _local_model = YOLO(CONFIG["keypoint_weights_path"])
        r = _local_model(image_path, verbose=False)[0]
        if r.keypoints is not None and len(r.boxes) > 0:
            best = int(r.boxes.conf.argmax())
            xy = r.keypoints.xy[best].cpu().numpy()
            cf = r.keypoints.conf[best].cpu().numpy() if r.keypoints.conf is not None else np.ones(len(xy))
            kps = [[float(x), float(y), float(c)] for (x, y), c in zip(xy, cf)]
    else:
        result = _get_client().infer(image_path, model_id=CONFIG["keypoint_model_id"])
        preds = result.get("predictions", [])
        if preds:
            best = max(preds, key=lambda p: p.get("confidence", 0))
            # IMPORTANT: index by each keypoint's own class_id (its canonical pitch-landmark id),
            # never by its position in the returned list. The API may omit low-confidence points
            # or return them out of canonical order; trusting list position silently mismatches
            # detected pixel positions to the wrong real-world pitch point (looks like a mirrored
            # or scrambled pitch, especially on partial/zoomed frames with few visible keypoints).
            kps = [[0.0, 0.0, 0.0] for _ in range(len(PITCH_VERTICES))]
            for k in best.get("keypoints", []):
                idx = k.get("class_id")
                if idx is None:
                    try:
                        idx = int(k.get("class_name", k.get("class", "")))
                    except (TypeError, ValueError):
                        continue
                if 0 <= idx < len(kps):
                    kps[idx] = [float(k["x"]), float(k["y"]), float(k.get("confidence", 0.0))]
    with open(cpath, "w") as f:
        json.dump(kps, f)
    return kps


def _hull_area(pts):
    pts = np.asarray(pts, dtype=np.float32)
    return float(cv2.contourArea(cv2.convexHull(pts))) if len(pts) >= 3 else 0.0


def compute_homography(kps, img_size):
    # Image px -> pitch meters. Returns (H, inlier keypoint ids, reason). H is None if calibration isn't trustworthy.
    iw, ih = img_size
    ids = [i for i, (x, y, c) in enumerate(kps) if c >= CONFIG["keypoint_conf"] and i < len(PITCH_VERTICES)]
    if len(ids) < CONFIG["min_keypoints"]:
        return None, ids, f"only {len(ids)} confident pitch keypoints (need {CONFIG['min_keypoints']})"
    img_pts = np.array([[kps[i][0], kps[i][1]] for i in ids], dtype=np.float32)
    pitch_pts = PITCH_VERTICES[ids]

    if _hull_area(pitch_pts) < CONFIG["min_pitch_hull_m2"] or _hull_area(img_pts) < 0.02 * iw * ih:
        return None, ids, "keypoints lie on (almost) a line - calibration would be unstable"

    H_p2i, mask = cv2.findHomography(pitch_pts, img_pts, cv2.RANSAC, CONFIG["ransac_px"])
    if H_p2i is None or mask is None:
        return None, ids, "homography failed"
    inl = mask.ravel().astype(bool)
    if inl.sum() < CONFIG["min_keypoints"]:
        return None, ids, f"only {int(inl.sum())} of {len(ids)} keypoints agree with each other (noisy detections)"
    if _hull_area(pitch_pts[inl]) < CONFIG["min_pitch_hull_m2"]:
        return None, ids, "agreeing keypoints lie on (almost) a line"

    H_p2i, _ = cv2.findHomography(pitch_pts[inl], img_pts[inl], 0)      # refit on inliers only
    if H_p2i is None:
        return None, ids, "homography failed"
    proj = cv2.perspectiveTransform(pitch_pts[inl].reshape(-1, 1, 2), H_p2i).reshape(-1, 2)
    rmse = float(np.sqrt(((proj - img_pts[inl]) ** 2).sum(axis=1).mean()))
    if rmse > CONFIG["ransac_px"]:
        return None, ids, f"reprojection error too high ({rmse:.1f}px)"
    try:
        H = np.linalg.inv(H_p2i)
    except np.linalg.LinAlgError:
        return None, ids, "singular homography"
    return H, [i for i, keep in zip(ids, inl) if keep], ""


def plausible_projection(xy):
    # Sanity check on where PEOPLE land after projection (catches wrong-but-confident calibrations).
    xy = np.asarray(xy, dtype=np.float64)
    xy = xy[np.isfinite(xy).all(axis=1)] if len(xy) else xy
    if len(xy) < 3:
        return False, "projection produced invalid positions"
    m = 8.0
    inside = float(((xy[:, 0] >= -m) & (xy[:, 0] <= L + m) & (xy[:, 1] >= -m) & (xy[:, 1] <= W + m)).mean())
    if inside < CONFIG["min_on_pitch_frac"]:
        return False, "most people project off the pitch (calibration looks wrong)"
    std = np.linalg.svd(xy - xy.mean(axis=0), compute_uv=False) / np.sqrt(len(xy))
    if std[1] < CONFIG["min_spread_m"]:
        return False, "projected people collapse onto a line (degenerate calibration)"
    return True, ""


def grass_quad_homography(img_rgb):
    """APPROX calibration: fit a 4-corner shape to the visible grass and map it to the whole pitch.
    Assumes the frame shows (roughly) the full pitch with the far touchline at the top.
    Returns (H image->pitch, quad in image px, reason)."""
    h, w = img_rgb.shape[:2]
    sc = 640.0 / max(h, w)
    small = cv2.resize(img_rgb, (max(1, int(w * sc)), max(1, int(h * sc))))
    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    mask = ((hsv[..., 0] >= 30) & (hsv[..., 0] <= 90) & (hsv[..., 1] >= 40) & (hsv[..., 2] >= 40)).astype(np.uint8) * 255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)   # bridges pitch lines and players
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None, None, "no grass found"
    cnt = max(cnts, key=cv2.contourArea)
    frac = cv2.contourArea(cnt) / float(mask.size)
    if frac < CONFIG["min_grass_frac"]:
        return None, None, f"grass covers only {frac:.0%} of the image"

    hull = cv2.convexHull(cnt)
    peri = cv2.arcLength(hull, True)
    quad = None
    for eps in np.linspace(0.01, 0.15, 30):
        approx = cv2.approxPolyDP(hull, eps * peri, True)
        if len(approx) == 4:
            quad = approx.reshape(4, 2).astype(np.float32)
            break
        if len(approx) < 4:
            break
    if quad is None:
        quad = cv2.boxPoints(cv2.minAreaRect(hull)).astype(np.float32)

    # order: top-left, top-right, bottom-right, bottom-left
    q = quad[np.argsort(quad[:, 1])]
    top, bot = q[:2][np.argsort(q[:2, 0])], q[2:][np.argsort(q[2:, 0])]
    quad = np.array([top[0], top[1], bot[1], bot[0]], dtype=np.float32)

    if cv2.contourArea(quad) / float(mask.size) < 0.15 or np.min(np.linalg.norm(quad - np.roll(quad, 1, axis=0), axis=1)) < 0.05 * max(small.shape[:2]):
        return None, None, "grass shape too small or degenerate"

    quad_full = quad / sc                      # back to ORIGINAL image pixels (H must use these, not the resized ones)
    dst = np.array([[0, 0], [L, 0], [L, W], [0, W]], dtype=np.float32)
    H = cv2.getPerspectiveTransform(quad_full.astype(np.float32), dst)
    return H, quad_full, ""


def project_points(H, pts_px):
    pts = np.array(pts_px, dtype=np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H).reshape(-1, 2)

## Team + goalkeeper assignment

In [ ]:
# Torso crop -> remove grass -> median Lab color -> KMeans(k=2). Cluster closer to pure red (in
# Lab chroma) is "red", the other "blue" — avoids the red-hue-wraparound bug of averaging raw HSV
# hue (a red cluster split across hue 0/179 can average out near green/blue and mislabel teams).
# Goalkeepers take the team of the nearest player, measured in PITCH METERS (needs pitch_x/pitch_y
# already set — call this AFTER get_pitch_positions, not before).
def torso_color_lab(img_rgb, bbox):
    H_img, W_img = img_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    bw, bh = x2 - x1, y2 - y1
    if bw < 4 or bh < 8:
        return None
    x1, x2 = x1 + int(0.2 * bw), x2 - int(0.2 * bw)
    ty1, ty2 = y1 + int(0.15 * bh), y1 + int(0.55 * bh)
    x1, x2 = max(0, x1), min(W_img, x2)
    ty1, ty2 = max(0, ty1), min(H_img, ty2)
    crop = img_rgb[ty1:ty2, x1:x2]
    if crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    green = (hsv[..., 0] >= 30) & (hsv[..., 0] <= 90) & (hsv[..., 1] >= 50) & (hsv[..., 2] >= 40)
    px = crop[~green]
    if len(px) < 20:
        px = crop.reshape(-1, 3)
    lab = cv2.cvtColor(px.reshape(-1, 1, 3).astype(np.uint8), cv2.COLOR_RGB2LAB).reshape(-1, 3)
    return np.median(lab.astype(np.float32), axis=0)


def assign_teams(img_rgb, detections, min_players):
    for d in detections:
        d["team"] = None

    players = [d for d in detections if d["class"] == "player"]
    keepers = [d for d in detections if d["class"] == "goalkeeper"]

    if len(players) >= min_players:
        feats = [torso_color_lab(img_rgb, d["bbox_px"]) for d in players]
        ok = [i for i, f in enumerate(feats) if f is not None]
        if len(ok) >= min_players:
            X = np.array([feats[i] for i in ok])
            km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
            RED_AB, BLUE_AB = np.array([80.0, 67.0]), np.array([79.0, -108.0])
            def redness(c):
                ab = X[km.labels_ == c][:, 1:3].mean(axis=0) - 128.0
                return np.linalg.norm(ab - BLUE_AB) - np.linalg.norm(ab - RED_AB)
            red_c = max(range(2), key=redness)
            for i, lab in zip(ok, km.labels_):
                players[i]["team"] = "red" if lab == red_c else "blue"
    for d in players:
        if d["team"] is None:
            d["team"] = "unknown"

    teamed = [d for d in players if d["team"] in ("red", "blue") and "pitch_x" in d]
    for gk in keepers:
        if not teamed or "pitch_x" not in gk:
            gk["team"] = "unknown"
            continue
        gx, gy = gk["pitch_x"], gk["pitch_y"]
        nearest = min(teamed, key=lambda p: (p["pitch_x"]-gx)**2 + (p["pitch_y"]-gy)**2)
        gk["team"] = nearest["team"]
    return detections

## Project detections onto the pitch
`get_pitch_positions` no longer clips out-of-range projections back onto the pitch boundary
(that hid bad calibrations by pinning wrong positions to the touchline). Instead, a detection
only gets `pitch_x`/`pitch_y` if it lands within `max_outside_m` of the real pitch; anything
further off is left unset and excluded downstream, same as the other notebooks.

In [ ]:
def get_pitch_positions(detections, H):
    """Mutates detections in place, adding pitch_x/pitch_y where the projection is plausible.
    Returns False if H is None (caller must skip this image entirely)."""
    if H is None:
        return False
    xy = project_points(H, [d["ground_point_px"] for d in detections])
    m = CONFIG["max_outside_m"]
    for d, (x, y) in zip(detections, xy):
        if -m <= x <= L + m and -m <= y <= W + m and np.isfinite(x) and np.isfinite(y):
            d["pitch_x"], d["pitch_y"] = float(x), float(y)
        else:
            d.pop("pitch_x", None); d.pop("pitch_y", None)
    return True

In [ ]:
# ============================================================
# Core geometry primitive — point-to-segment distance. Everything else depends on this being
# correct, so it's verified with hand-checked assertions before anything else runs.
# ============================================================
def point_to_segment_distance(P, T, D):
    """Distance from point D to the segment P-T, plus the clamped projection parameter t
    (0 = at P, 1 = at T) and the closest point on the segment."""
    P, T, D = np.array(P, dtype=float), np.array(T, dtype=float), np.array(D, dtype=float)
    seg = T - P
    seg_len_sq = np.dot(seg, seg)
    if seg_len_sq == 0:
        return float(np.linalg.norm(D - P)), 0.0, P
    t = np.dot(D - P, seg) / seg_len_sq
    t_clamped = np.clip(t, 0.0, 1.0)
    closest = P + t_clamped * seg
    distance = float(np.linalg.norm(D - closest))
    return distance, float(t_clamped), closest


# --- Verification cell: run this and confirm no AssertionError before trusting anything below ---
def _assert_close(a, b, tol=1e-6):
    assert abs(a - b) < tol, f"expected {b}, got {a}"

# Defender exactly on the line, halfway
d, t, cp = point_to_segment_distance((0, 0), (10, 0), (5, 0))
_assert_close(d, 0.0); _assert_close(t, 0.5)

# Defender perpendicular to the midpoint
d, t, cp = point_to_segment_distance((0, 0), (10, 0), (5, 3))
_assert_close(d, 3.0); _assert_close(t, 0.5)

# Defender past the T endpoint — must clamp to t=1, not extrapolate the infinite line
d, t, cp = point_to_segment_distance((0, 0), (10, 0), (15, 0))
_assert_close(d, 5.0); _assert_close(t, 1.0)

# Defender before the P endpoint — must clamp to t=0
d, t, cp = point_to_segment_distance((0, 0), (10, 0), (-5, 0))
_assert_close(d, 5.0); _assert_close(t, 0.0)

print("point_to_segment_distance: all checks passed.")

In [ ]:
# ============================================================
# Ground passing lanes.
# ============================================================
def build_lane(passer, receiver, opponents, interception_radius, t_range):
    min_dist = float("inf")
    nearest_defender = None
    for opp in opponents:
        d, t, closest = point_to_segment_distance(
            (passer["pitch_x"], passer["pitch_y"]),
            (receiver["pitch_x"], receiver["pitch_y"]),
            (opp["pitch_x"], opp["pitch_y"]),
        )
        if t < t_range[0] or t > t_range[1]:
            continue  # just marking passer/receiver personally, not cutting the lane itself
        if d < min_dist:
            min_dist = d
            nearest_defender = opp
            closest_point = closest

    pass_distance = float(np.hypot(receiver["pitch_x"] - passer["pitch_x"],
                                    receiver["pitch_y"] - passer["pitch_y"]))

    if nearest_defender is None:
        return {
            "receiver_position": [receiver["pitch_x"], receiver["pitch_y"]],
            "distance_m": pass_distance,
            "min_defender_distance_m": None,
            "blocked": False,
            "nearest_defender": None,
            "defender_shift_needed_m": None,
        }

    blocked = min_dist < interception_radius
    return {
        "receiver_position": [receiver["pitch_x"], receiver["pitch_y"]],
        "distance_m": pass_distance,
        "min_defender_distance_m": float(min_dist),
        "blocked": blocked,
        "nearest_defender": {"team": nearest_defender.get("team"),
                              "position": [nearest_defender["pitch_x"], nearest_defender["pitch_y"]]},
        "defender_shift_needed_m": float(min_dist) if not blocked else 0.0,
    }


def compute_ground_lanes(passer, teammates, opponents, config):
    return [
        build_lane(passer, mate, opponents, config["interception_radius_m"], config["lane_t_range"])
        for mate in teammates
    ]

## Lofted alternatives (contest radius now scales with pass distance)

In [ ]:
def landing_pressure(passer, receiver, opponents, config):
    pass_distance = float(np.hypot(receiver["pitch_x"] - passer["pitch_x"],
                                    receiver["pitch_y"] - passer["pitch_y"]))
    contest_radius = config["contest_radius_base_m"] + config["contest_radius_per_10m"] * max(0.0, pass_distance - 20.0) / 10.0
    nearby = [o for o in opponents
              if np.hypot(o["pitch_x"] - receiver["pitch_x"], o["pitch_y"] - receiver["pitch_y"]) <= contest_radius]
    n = len(nearby)
    level = "low" if n == 0 else ("moderate" if n == 1 else "high")
    return {"opponents_near_landing": n, "landing_pressure": level, "contest_radius_m": contest_radius}


def compute_lofted_alternatives(passer, ground_lanes, teammates, opponents, config):
    alternatives = []
    for lane, mate in zip(ground_lanes, teammates):
        if not lane["blocked"]:
            continue
        pressure = landing_pressure(passer, mate, opponents, config)
        alternatives.append({
            "receiver_position": [mate["pitch_x"], mate["pitch_y"]],
            "reason": "ground_lane_blocked",
            **pressure,
        })
    return alternatives

## Through balls (fixed: offside line now uses the correct end of the defense)
`get_defensive_line_x` previously always took `min(x)` regardless of attack direction. For a
team attacking in the `increasing_x` direction, that grabbed the **most advanced** defender
(closest to halfway) instead of the **deepest** one — so runners were flagged "beyond the
defensive line" as soon as they passed the first defender, not the real backline. Fixed to use
`max(x)` for `increasing_x` and `min(x)` for `decreasing_x`.

In [ ]:
def get_defensive_line_x(opponents_outfield, direction):
    if not opponents_outfield:
        return None
    xs = [p["pitch_x"] for p in opponents_outfield]
    return float(max(xs)) if direction == "increasing_x" else float(min(xs))


def compute_through_balls(passer, teammates, opponents, config, passer_team):
    direction = config["attack_direction"].get(passer_team)
    if direction is None:
        return None  # omitted entirely, not an empty list

    opponents_outfield = [o for o in opponents if o["class"] == "player"]
    defensive_line_x = get_defensive_line_x(opponents_outfield, direction)
    if defensive_line_x is None:
        return []

    def is_beyond_line(x):
        return x > defensive_line_x if direction == "increasing_x" else x < defensive_line_x

    through_balls = []
    for runner in teammates:
        if is_beyond_line(runner["pitch_x"]):
            continue

        target_x = defensive_line_x + (2.0 if direction == "increasing_x" else -2.0)
        target_point = (target_x, runner["pitch_y"])

        min_dist_to_lane = float("inf")
        for opp in opponents:
            d, t, _ = point_to_segment_distance(
                (passer["pitch_x"], passer["pitch_y"]), target_point,
                (opp["pitch_x"], opp["pitch_y"]),
            )
            if config["lane_t_range"][0] <= t <= config["lane_t_range"][1]:
                min_dist_to_lane = min(min_dist_to_lane, d)
        lane_clear = min_dist_to_lane > config["interception_radius_m"]

        runner_dist = float(np.hypot(runner["pitch_x"] - target_point[0], runner["pitch_y"] - target_point[1]))
        nearest_opp_dist = min(
            (float(np.hypot(o["pitch_x"] - target_point[0], o["pitch_y"] - target_point[1])) for o in opponents),
            default=float("inf"),
        )
        race_margin = nearest_opp_dist - runner_dist

        through_balls.append({
            "runner_position": [runner["pitch_x"], runner["pitch_y"]],
            "target_point": list(target_point),
            "lane_clear": lane_clear,
            "race_margin_m": race_margin,
            "offside_risk": False,
        })

    for runner in teammates:
        if is_beyond_line(runner["pitch_x"]):
            through_balls.append({
                "runner_position": [runner["pitch_x"], runner["pitch_y"]],
                "target_point": None,
                "lane_clear": None,
                "race_margin_m": None,
                "offside_risk": True,
            })

    return through_balls

## Orchestration

In [ ]:
def select_passer(detections, selection_mode):
    if selection_mode == "nearest_to_ball":
        ball = next((d for d in detections if d["class"] == "ball" and "pitch_x" in d), None)
        people = [d for d in detections if d["class"] in ("player", "goalkeeper")
                  and d.get("team") in ("red", "blue") and "pitch_x" in d]
        if ball is None or not people:
            return None
        dists = [np.hypot(p["pitch_x"] - ball["pitch_x"], p["pitch_y"] - ball["pitch_y"]) for p in people]
        return people[int(np.argmin(dists))]
    else:
        team, idx = selection_mode
        candidates = [d for d in detections if d["class"] in ("player", "goalkeeper")
                      and d.get("team") == team and "pitch_x" in d]
        return candidates[idx] if idx < len(candidates) else None


def process_image_passing_analysis(image_path, detections, config):
    person_count = sum(1 for d in detections if d["class"] in PERSON_CLASSES)
    if person_count < config["min_person_count"]:
        return None, "too_few_people", None

    image_pil = Image.open(image_path).convert("RGB")
    img_rgb = np.array(image_pil)

    overlay = {}
    if config["calibration"] == "keypoints":
        kps = get_keypoints(image_path)
        H, kp_ids, why = compute_homography(kps, (img_rgb.shape[1], img_rgb.shape[0]))
        overlay = {"kps": kps, "ids": kp_ids}
    else:
        H, quad, why = grass_quad_homography(img_rgb)
        overlay = {"quad": quad}
    if H is None:
        return None, f"homography_failed: {why}", overlay
    overlay["H"] = H

    if not get_pitch_positions(detections, H):
        return None, "homography_failed", overlay

    if config["calibration"] == "keypoints":
        people_xy = [(d["pitch_x"], d["pitch_y"]) for d in detections if d["class"] in PERSON_CLASSES and "pitch_x" in d]
        good, why = plausible_projection(people_xy)
        if not good:
            return None, f"implausible_projection: {why}", overlay

    detections = assign_teams(img_rgb, detections, config["min_players_for_team_split"])

    passer = select_passer(detections, config["passer_selection"])
    if passer is None:
        return None, "no_passer_available", overlay

    passer_team = passer.get("team")
    if passer_team not in ("red", "blue"):
        return None, "passer_team_unknown", overlay

    teammates = [d for d in detections if d.get("team") == passer_team and d is not passer
                 and d["class"] in ("player", "goalkeeper") and "pitch_x" in d]
    opponents = [d for d in detections if d.get("team") not in (passer_team, None)
                 and d["class"] in ("player", "goalkeeper") and "pitch_x" in d]

    if len(teammates) < 1:
        return None, "no_teammates_available", overlay

    ground_lanes = compute_ground_lanes(passer, teammates, opponents, config)
    lofted_alternatives = compute_lofted_alternatives(passer, ground_lanes, teammates, opponents, config)
    through_balls = compute_through_balls(passer, teammates, opponents, config, passer_team)

    result = {
        "image_path": image_path,
        "passer": {"team": passer_team, "position": [passer["pitch_x"], passer["pitch_y"]]},
        "ground_lanes": ground_lanes,
        "lofted_alternatives": lofted_alternatives,
        "through_balls": through_balls,
        "attack_direction_known": through_balls is not None,
    }
    return result, None, overlay

## Visualization

In [ ]:
PENALTY_DEPTH, PENALTY_WIDTH = PB_L, PB_W    # from the pitch-calibration cell (matches the keypoint model's pitch)
GOAL_AREA_DEPTH, GOAL_AREA_WIDTH = GB_L, GB_W
PITCH_W, PITCH_H = L, W

def draw_pitch(ax):
    ax.add_patch(mpatches.Rectangle((0, 0), PITCH_W, PITCH_H, facecolor="black",
                                     edgecolor="white", linewidth=2, zorder=0))
    ax.plot([PITCH_W / 2, PITCH_W / 2], [0, PITCH_H], color="white", linewidth=1, zorder=1)
    ax.add_patch(mpatches.Circle((PITCH_W / 2, PITCH_H / 2), CC_R, fill=False,
                                  edgecolor="white", linewidth=1, zorder=1))
    py_penalty = (PITCH_H - PENALTY_WIDTH) / 2
    py_goal_area = (PITCH_H - GOAL_AREA_WIDTH) / 2
    for x_start, direction in [(0, 1), (PITCH_W, -1)]:
        px = x_start if direction == 1 else x_start - PENALTY_DEPTH
        ax.add_patch(mpatches.Rectangle((px, py_penalty), PENALTY_DEPTH, PENALTY_WIDTH,
                                         fill=False, edgecolor="white", linewidth=1, zorder=1))
        gx = x_start if direction == 1 else x_start - GOAL_AREA_DEPTH
        ax.add_patch(mpatches.Rectangle((gx, py_goal_area), GOAL_AREA_DEPTH, GOAL_AREA_WIDTH,
                                         fill=False, edgecolor="white", linewidth=1, zorder=1))
    ax.set_xlim(0, PITCH_W)
    ax.set_ylim(0, PITCH_H)
    ax.set_aspect("equal")
    ax.axis("off")


def draw_players_and_passer(detections, passer_position, ax):
    draw_pitch(ax)
    for d in detections:
        if "pitch_x" not in d:
            continue
        color = {"ball": "white", "referee": "yellow"}.get(
            d["class"], {"red": "red", "blue": "blue"}.get(d.get("team"), "gray"))
        marker = "*" if d["class"] == "ball" else ("^" if d["class"] == "referee" else "o")
        size = 220 if d["class"] == "ball" else 80
        ax.scatter(d["pitch_x"], d["pitch_y"], color=color, marker=marker, s=size,
                   edgecolor="white", linewidth=0.6, zorder=4)
    px, py = passer_position
    ax.scatter(px, py, s=250, facecolors="none", edgecolors="lime", linewidth=2, zorder=6)


def plot_ground_passes(result, detections, ax):
    """Image 1: ground passing lanes only. Green = open, red = blocked, orange dashed =
    nearest defender's position relative to where they'd need to be to block an open lane."""
    px, py = result["passer"]["position"]
    draw_players_and_passer(detections, (px, py), ax)

    for lane in result["ground_lanes"]:
        rx, ry = lane["receiver_position"]
        color = "red" if lane["blocked"] else "limegreen"
        alpha = 0.9 if lane["blocked"] else min(1.0, 0.3 + (lane["min_defender_distance_m"] or 5) / 10)
        ax.plot([px, rx], [py, ry], color=color, linewidth=2, alpha=alpha, zorder=3)
        if not lane["blocked"] and lane["nearest_defender"]:
            dx, dy = lane["nearest_defender"]["position"]
            ax.plot([dx, rx], [dy, ry], color="orange", linewidth=1, linestyle="--", zorder=3)

    ax.set_title("GROUND PASSES — green=open, red=blocked, orange dashed=how defense closes it",
                 fontsize=8, color="black")


def plot_lofted_passes(result, detections, ax):
    """Image 2: lofted alternatives only, styled distinctly from ground passes (dashed
    magenta line, since these exist specifically because the ground lane is blocked — not
    meant to be confused with an open ground lane). Landing-zone pressure shown as a colored
    ring around the receiver: green=low, orange=moderate, red=high pressure on arrival."""
    px, py = result["passer"]["position"]
    draw_players_and_passer(detections, (px, py), ax)

    pressure_colors = {"low": "limegreen", "moderate": "orange", "high": "red"}
    if not result["lofted_alternatives"]:
        ax.set_title("LOFTED ALTERNATIVES — none (no blocked ground lanes to route around)",
                     fontsize=8, color="black")
        return

    for lofted in result["lofted_alternatives"]:
        rx, ry = lofted["receiver_position"]
        ax.plot([px, rx], [py, ry], color="magenta", linewidth=2, linestyle="--",
                alpha=0.85, zorder=3)
        ring_color = pressure_colors[lofted["landing_pressure"]]
        ax.add_patch(mpatches.Circle((rx, ry), radius=3.0, fill=False,
                                      edgecolor=ring_color, linewidth=2.5, zorder=5))

    ax.set_title("LOFTED ALTERNATIVES (dashed magenta) — ring color = landing pressure: "
                 "green=low, orange=moderate, red=high", fontsize=8, color="black")


def plot_through_balls(result, detections, ax):
    """Image 3: through balls only. Cyan solid = clear lane, cyan dotted = lane not clear,
    gray dashed with an X marker = runner already beyond the defensive line (offside risk,
    not a valid run to suggest)."""
    px, py = result["passer"]["position"]
    draw_players_and_passer(detections, (px, py), ax)

    through_balls = result["through_balls"]
    if through_balls is None:
        ax.set_title("THROUGH BALLS — not evaluated (attack_direction not configured for this team)",
                     fontsize=8, color="black")
        return
    if not through_balls:
        ax.set_title("THROUGH BALLS — none available", fontsize=8, color="black")
        return

    for tb in through_balls:
        rx, ry = tb["runner_position"]
        if tb["offside_risk"] or tb["target_point"] is None:
            ax.plot([px, rx], [py, ry], color="gray", linewidth=1.5, linestyle="--", zorder=3)
            ax.scatter(rx, ry, marker="x", color="gray", s=120, zorder=6)
            continue
        tx, ty = tb["target_point"]
        style = "-" if tb["lane_clear"] else ":"
        ax.annotate("", xy=(tx, ty), xytext=(px, py),
                   arrowprops=dict(arrowstyle="->", color="cyan", linewidth=2, linestyle=style))

    ax.set_title("THROUGH BALLS — cyan solid=clear lane, cyan dotted=lane blocked, "
                 "gray X=already offside position", fontsize=8, color="black")

## Calibration check (verification only)
Before trusting the pass geometry, check that the pitch calibration is actually right: this
draws the pitch model projected back onto the real photo (cyan). If the cyan lines sit on the
real white lines, the homography for that frame is trustworthy.

In [ ]:
def pitch_polylines():
    lines = [np.array([[0, 0], [L, 0], [L, W], [0, W], [0, 0]], float),
             np.array([[L/2, 0], [L/2, W]], float)]
    for x0, sgn in [(0, 1), (L, -1)]:
        bx, gx = x0 + sgn * PB_L, x0 + sgn * GB_L
        lines.append(np.array([[x0, (W-PB_W)/2], [bx, (W-PB_W)/2], [bx, (W+PB_W)/2], [x0, (W+PB_W)/2]], float))
        lines.append(np.array([[x0, (W-GB_W)/2], [gx, (W-GB_W)/2], [gx, (W+GB_W)/2], [x0, (W+GB_W)/2]], float))
    t = np.linspace(0, 2 * np.pi, 90)
    lines.append(np.c_[L/2 + CC_R * np.cos(t), W/2 + CC_R * np.sin(t)])
    return lines


def densify(pl, step=1.0):
    out = []
    for a, b in zip(pl[:-1], pl[1:]):
        n = max(2, int(np.linalg.norm(b - a) / step))
        out.append(np.linspace(a, b, n, endpoint=False))
    out.append(pl[-1:])
    return np.vstack(out)


def draw_pitch_on_image(ax, H, img_shape, anchor_pitch_pts=None):
    # Project the pitch model back onto the photo. If calibration is right, cyan lines sit on the real white lines.
    ih, iw = img_shape[:2]
    Hinv = np.linalg.inv(H)
    if anchor_pitch_pts is not None and len(anchor_pitch_pts):      # pitch points known to be in front of the camera
        a = np.asarray(anchor_pitch_pts, float)
        ref = np.sign(np.median((Hinv @ np.c_[a, np.ones(len(a))].T)[2])) or 1.0
    else:
        c = H @ np.array([iw / 2, ih / 2, 1.0])
        c = c[:2] / c[2]
        ref = np.sign((Hinv @ np.array([c[0], c[1], 1.0]))[2]) or 1.0
    for pl in pitch_polylines():
        pts = densify(pl)
        hp = (Hinv @ np.c_[pts, np.ones(len(pts))].T).T
        ok = (hp[:, 2] * ref) > 1e-6
        xy = hp[:, :2] / np.where(ok, hp[:, 2], 1.0)[:, None]
        ok &= (xy[:, 0] > -iw) & (xy[:, 0] < 2 * iw) & (xy[:, 1] > -ih) & (xy[:, 1] < 2 * ih)
        xy[~ok] = np.nan
        ax.plot(xy[:, 0], xy[:, 1], "-", color="cyan", lw=1.3, alpha=0.9)
    ax.set_xlim(0, iw)
    ax.set_ylim(ih, 0)

In [ ]:
def print_result_summary(fname, result):
    """Text form of exactly what each plot is showing - printed so the numbers behind the
    plots (not just green/red lines) are visible and easy to copy out or log."""
    p = result["passer"]
    print(f"\n{fname}  |  passer team={p['team']}  pos=({p['position'][0]:.1f}, {p['position'][1]:.1f})")

    print("  GROUND LANES:")
    for lane in result["ground_lanes"]:
        rx, ry = lane["receiver_position"]
        status = "BLOCKED" if lane["blocked"] else "open"
        dist = lane["min_defender_distance_m"]
        dist_txt = f", nearest defender {dist:.1f}m from lane" if dist is not None else ""
        print(f"    -> ({rx:.1f}, {ry:.1f}): {status}{dist_txt}")

    if not result["lofted_alternatives"]:
        print("  LOFTED ALTERNATIVES: none (no blocked ground lanes to route around)")
    else:
        print("  LOFTED ALTERNATIVES:")
        for lofted in result["lofted_alternatives"]:
            rx, ry = lofted["receiver_position"]
            print(f"    -> ({rx:.1f}, {ry:.1f}): {lofted['landing_pressure']} pressure "
                  f"({lofted['opponents_near_landing']} opponents within {lofted['contest_radius_m']:.1f}m)")

    tb = result["through_balls"]
    if tb is None:
        print("  THROUGH BALLS: not evaluated (attack_direction not configured for this team - "
              "set CONFIG['attack_direction'] in the config cell if you want this)")
    elif not tb:
        print("  THROUGH BALLS: none available")
    else:
        print("  THROUGH BALLS:")
        for t in tb:
            rx, ry = t["runner_position"]
            if t["offside_risk"]:
                print(f"    -> ({rx:.1f}, {ry:.1f}): already beyond the defensive line (offside risk, not suggested)")
            else:
                clear = "clear" if t["lane_clear"] else "NOT clear"
                print(f"    -> ({rx:.1f}, {ry:.1f}): lane {clear}, race margin {t['race_margin_m']:+.1f}m "
                      f"({'runner ahead' if t['race_margin_m'] > 0 else 'defender ahead'})")


def show_calibration_check(image_path, overlay, title, ax):
    """Draws into the given axis (part of a combined 2-up figure)."""
    img_rgb = np.array(Image.open(image_path).convert("RGB"))
    ax.imshow(img_rgb)
    if CONFIG["show_reprojection"] and overlay.get("H") is not None:
        draw_pitch_on_image(ax, overlay["H"], img_rgb.shape,
                             [PITCH_VERTICES[i] for i in overlay.get("ids", [])] if "ids" in overlay else None)
    elif overlay.get("quad") is not None:
        q = np.vstack([overlay["quad"], overlay["quad"][:1]])
        ax.plot(q[:, 0], q[:, 1], "-", color="lime", lw=2)
    ax.set_title(f"{title}  |  calibration check: cyan should sit on the real pitch lines", fontsize=8)
    ax.axis("off")


def run_verification(name, metadata_path, n_images):
    """Two figures per image, two plots each, instead of four separate full-width figures:
    [calibration check | ground passes], then [lofted alternatives | through balls]."""
    records = load_metadata(metadata_path)[:n_images * 3]
    shown = 0
    for record in records:
        if shown >= n_images:
            break
        detections_copy = json.loads(json.dumps(record["detections"]))
        result, skip_reason, overlay = process_image_passing_analysis(record["image_path"], detections_copy, CONFIG)
        fname = os.path.basename(record["image_path"])
        if result is None:
            print(f"Skipped {fname}: {skip_reason}")
            continue

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
        show_calibration_check(record["image_path"], overlay, fname, ax1)
        plot_ground_passes(result, detections_copy, ax2)
        plt.tight_layout(); plt.show()

        fig, (ax3, ax4) = plt.subplots(1, 2, figsize=(15, 5.5))
        plot_lofted_passes(result, detections_copy, ax3)
        plot_through_balls(result, detections_copy, ax4)
        plt.suptitle(fname, fontsize=9, y=1.02); plt.tight_layout(); plt.show()

        print_result_summary(fname, result)

        shown += 1

    if shown == 0:
        print(f"No images could be visualized for {name} - check skip reasons above.")


for name, path in CONFIG["metadata_paths"].items():
    print(f"\n--- Verification: {name} ---")
    run_verification(name, path, CONFIG["num_verification_images"])

In [ ]:
def load_already_done(output_path):
    done = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    done.add(json.loads(line)["image_path"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return done


def process_metadata_batch(name, metadata_path):
    records = load_metadata(metadata_path)
    output_path = os.path.join(CONFIG["output_dir"], f"{name}_passing_lanes.jsonl")
    already_done = load_already_done(output_path)

    print(f"\n--- {name}: {len(records)} records, {len(already_done)} already done ---")

    kept, skip_counts = 0, {}
    fail_streak = 0
    with open(output_path, "a") as out_f:
        for record in records:
            image_path = record["image_path"]
            if image_path in already_done:
                continue

            try:
                result, skip_reason, _ = process_image_passing_analysis(image_path, record["detections"], CONFIG)
                fail_streak = 0
            except RuntimeError as e:      # keypoint source not configured: stop, don't spam per image
                print(f"STOPPING: {e}")
                return
            except Exception as e:
                result, skip_reason = None, f"error: {e}"
                fail_streak += 1
                if fail_streak >= 3:
                    print("STOPPING: 3 failures in a row - check calibration source, API key, internet access.")
                    return

            if result is None:
                skip_counts[skip_reason] = skip_counts.get(skip_reason, 0) + 1
                continue

            out_f.write(json.dumps(result) + "\n")
            out_f.flush()
            kept += 1

    print(f"{name}: wrote {kept} new records. Skip reasons: {skip_counts}")


for name, path in CONFIG["metadata_paths"].items():
    process_metadata_batch(name, path)

## Notes

- **Always run the verification cell first**, and check the cyan calibration-check overlay
  before trusting any pass geometry — if the cyan lines don't sit on the real pitch lines,
  nothing downstream can be trusted for that frame.
- **Ground-pass blocking uses perpendicular distance from each defender to the pass line
  segment** (`point_to_segment_distance`, verified by the assertions), clamped so defenders past
  either end don't count. This was already correct and is unchanged.
- **Lofted contest radius now scales with pass distance** (`contest_radius_base_m` +
  `contest_radius_per_10m` per 10m over 20m), since a longer lofted ball hangs longer and gives
  defenders more time to converge on the landing spot.
- **Through-ball offside line fixed**: now uses the deepest defender in the direction of attack
  (`max(x)` for `increasing_x`, `min(x)` for `decreasing_x`), not always `min(x)`. The old
  version flagged runners as "beyond the line" as soon as they passed the *first* defender for
  teams attacking in the increasing-x direction.
- **Skip reasons are logged, not silent** — `too_few_people`, `homography_failed: <reason>`,
  `implausible_projection: <reason>`, `no_passer_available`, `passer_team_unknown`,
  `no_teammates_available`. Check the printed counts after a batch run.
- **`attack_direction_known: false`** means through-ball analysis was skipped for that image's
  passer team — not that no through balls exist.
- **Lofted and through-ball outputs are single-frame geometric proxies**, not physics or
  player-speed simulations — there is no velocity information in one image.
- Keypoint results are cached in `./keypoint_cache`, so re-running doesn't re-call the API.